# SparseLoCo + LoRA vs Baseline — Fine-tuning Distribuído Simulado

Comparação entre duas abordagens de fine-tuning no benchmark **MedMCQA** (domínio biomédico):

| Abordagem | Parâmetros treináveis | Comunicação |
|---|---|---|
| **Baseline** — AdamW + LoRA centralizado | LoRA adapters | sem compressão |
| **SparseLoCo + LoRA** — distribuído simulado | LoRA adapters | Top-k comprimido |

**Pergunta de pesquisa:** O fine-tuning com SparseLoCo+LoRA atinge accuracy comparável ao LoRA centralizado? Se sim, o SparseLoCo viabiliza fine-tuning distribuído sem degradar a qualidade do modelo.

> **Modelo de desenvolvimento:** `meta-llama/Llama-3.2-3B` (local/Colab T4)
> **Experimento final:** `1Covenant/Covenant-72B` (requer A100 80GB)

In [ ]:
!pip install -q torch transformers peft datasets matplotlib tqdm scikit-learn bitsandbytes>=0.46.1 accelerate

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Hiperparâmetros

In [ ]:
# Modelo
# Desenvolvimento local / Colab T4:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# Experimento final (requer A100 80GB):
# MODEL_ID = "1Covenant/Covenant-72B"

# Distribuição
R        = 4      # workers simulados
H        = 30     # inner steps por worker por round
T        = 20     # outer rounds

# Modelo e dados
B        = 4      # batch size
MAX_LEN  = 1024   # cobre questões médicas completas
N_LABELS = 5      # opções A, B, C, D, E

# LoRA
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# Otimizadores
LR_INNER = 2e-4
LR_OUTER = 1.0
BETA     = 0.95
TOPK_K   = 0.10

# Checkpoints (persiste entre sessões via Google Drive)
from google.colab import drive; drive.mount('/content/drive')
CKPT_DIR = "/content/drive/MyDrive/checkpoints"
# CKPT_DIR = "/content/checkpoints"
import os; os.makedirs(CKPT_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)

## 2. Dataset — MedQA (Domínio Biomédico)

MedQA é um benchmark de questões do exame médico americano (USMLE). Formato instruction-following com pergunta e 4 opções — a resposta é a letra correta (A, B, C, D).

- Input: instruction + questão com opções
- Labels: A→0, B→1, C→2, D→3 | Chance = 25%

In [ ]:
from sklearn.model_selection import train_test_split

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

raw = load_dataset("medalpaca/medical_meadow_medqa", split="train")
print(f"Total de amostras: {len(raw)}")

LABEL_MAP = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

# Filtra amostras com resposta válida (A-E)
all_samples = [d for d in raw if str(d["output"]).strip()[0].upper() in LABEL_MAP]
print(f"Amostras válidas: {len(all_samples)}")

train_val, test_data = train_test_split(all_samples, test_size=1000, random_state=SEED)
train_data, val_data = train_test_split(train_val,   test_size=1000, random_state=SEED)

print(f"Treino: {len(train_data)} | Validação: {len(val_data)} | Teste: {len(test_data)}")

class MedQADataset(Dataset):
    def __init__(self, data):
        self.samples = data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        item = self.samples[i]
        text = f"{item['instruction']} {item['input']}"
        enc = tok(text, truncation=True, max_length=MAX_LEN,
                  padding="max_length", return_tensors="pt")
        label = LABEL_MAP[str(item["output"]).strip()[0].upper()]
        return enc["input_ids"].squeeze(), enc["attention_mask"].squeeze(), \
               torch.tensor(label)

train_ds = MedQADataset(train_data)
val_ds   = MedQADataset(val_data)
test_ds  = MedQADataset(test_data)

per = len(train_ds) // R
shards = [
    DataLoader(Subset(train_ds, range(r*per, (r+1)*per)),
               batch_size=B, shuffle=True, drop_last=True)
    for r in range(R)
]
central_loader = DataLoader(train_ds, batch_size=B, shuffle=True, drop_last=True)
val_loader     = DataLoader(val_ds,   batch_size=B, shuffle=False)
test_loader    = DataLoader(test_ds,  batch_size=B, shuffle=False)

print(f"Chance level: {1/N_LABELS:.0%}")

## 3. Modelo — TinyLlama-1.1B + LoRA para Classificação

TinyLlama (1.1B parâmetros) com arquitetura LLaMA idêntica ao Covenant-72B — usado para desenvolvimento local.

- **CPU:** carregado em FP32, sem quantização (bitsandbytes requer CUDA)
- **GPU (Colab):** carregado com QLoRA 4-bit para economizar VRAM

Para o experimento final, troque `MODEL_ID` para `1Covenant/Covenant-72B` (requer A100 80GB).

In [ ]:
def make_model():
    if DEVICE == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        base = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID,
            num_labels=N_LABELS,
            quantization_config=bnb_config,
            device_map="auto",
        )
        base = prepare_model_for_kbit_training(base)
    else:
        base = AutoModelForSequenceClassification.from_pretrained(
            MODEL_ID,
            num_labels=N_LABELS,
        )
        base = base.to(DEVICE)

    base.config.pad_token_id = tok.pad_token_id

    lora = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
    )
    return get_peft_model(base, lora)

def lora_params(m):
    return [p for p in m.parameters() if p.requires_grad]

def to_vec(params):
    return torch.cat([p.detach().cpu().float().flatten() for p in params])

def load_vec(v, params):
    off = 0
    for p in params:
        n = p.numel()
        p.data.copy_(v[off:off+n].view(p.shape).to(p.device).to(p.dtype))
        off += n

m_test = make_model()
m_test.print_trainable_parameters()

## 4. Funções de Treino e Avaliação

In [ ]:
def step_loss(m, ids, mask, labels):
    out = m(input_ids=ids.to(DEVICE),
            attention_mask=mask.to(DEVICE),
            labels=labels.to(DEVICE))
    return out.loss

@torch.no_grad()
def evaluate(m, loader):
    m.eval()
    preds, targets = [], []
    for ids, mask, labels in loader:
        out = m(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE))
        preds.extend(out.logits.argmax(-1).cpu().tolist())
        targets.extend(labels.tolist())
    m.train()
    return accuracy_score(targets, preds)

def topk_sparsify(v, k):
    out = torch.zeros_like(v)
    idx = v.abs().topk(k).indices
    out[idx] = v[idx]
    return out

## 5. Treinamento — SparseLoCo + LoRA

Cada round:
1. R workers treinam H steps localmente a partir dos parâmetros globais
2. Pseudo-gradiente: Δᵣ = θ_global − θ_local
3. Error Feedback + Top-k compressão
4. Outer update: θ_global ← θ_global − α · média(Δ̂ᵣ)

In [ ]:
def run_sparseloco(m, shards, val_loader):
    ps = lora_params(m)
    d  = sum(p.numel() for p in ps)
    k  = max(1, int(d * TOPK_K))

    ckpt_path = f"{CKPT_DIR}/sparseloco_ckpt.pt"

    # Tenta retomar de checkpoint
    if os.path.exists(ckpt_path):
        print(f"Retomando checkpoint: {ckpt_path}")
        ckpt       = torch.load(ckpt_path, map_location="cpu")
        start_t    = ckpt["round"] + 1
        load_vec(ckpt["lora_params"], ps)
        ef         = ckpt["ef"]
        opt_states = ckpt["opt_states"]
        history    = ckpt["history"]
        print(f"Continuando do round {start_t}/{T}")
    else:
        start_t    = 0
        ef         = [torch.zeros(d) for _ in range(R)]
        opt_states = [None] * R
        history    = []

    opts = [torch.optim.AdamW(ps, lr=LR_INNER) for _ in range(R)]
    print(f"LoRA params: {d:,} | Top-k: k={k} ({TOPK_K*100:.0f}%)")

    for t in tqdm(range(start_t, T), desc="SparseLoCo"):
        g_vec      = to_vec(ps)
        local_vecs = []
        round_loss = 0.0

        for r in range(R):
            load_vec(g_vec.clone(), ps)
            if opt_states[r] is not None:
                opts[r].load_state_dict(opt_states[r])
            it = iter(shards[r])
            for _ in range(H):
                try:    ids, mask, labels = next(it)
                except: it = iter(shards[r]); ids, mask, labels = next(it)
                opts[r].zero_grad()
                loss = step_loss(m, ids, mask, labels)
                loss.backward()
                opts[r].step()
                round_loss += loss.item()
            opt_states[r] = opts[r].state_dict()
            local_vecs.append(to_vec(ps))

        agg = torch.zeros(d)
        for r, lv in enumerate(local_vecs):
            pg     = g_vec - lv
            ef[r]  = BETA * ef[r] + pg
            comp   = topk_sparsify(ef[r], k)
            ef[r] -= comp
            agg   += comp
        agg /= R

        load_vec(g_vec - LR_OUTER * agg, ps)

        acc = evaluate(m, val_loader)
        history.append({"train_loss": round_loss / (R * H), "val_acc": acc})
        tqdm.write(f"  round={t:2d}  loss={history[-1]['train_loss']:.4f}  val_acc={acc:.4f}")

        # Salva checkpoint após cada round
        torch.save({
            "round":      t,
            "lora_params": to_vec(ps),
            "ef":          ef,
            "opt_states":  opt_states,
            "history":     history,
        }, ckpt_path)

    return history

## 6. Treinamento — Baseline (AdamW + LoRA centralizado)

In [ ]:
def run_baseline(m, loader, val_loader):
    ps  = lora_params(m)

    ckpt_path = f"{CKPT_DIR}/baseline_ckpt.pt"

    if os.path.exists(ckpt_path):
        print(f"Retomando checkpoint: {ckpt_path}")
        ckpt    = torch.load(ckpt_path, map_location="cpu")
        start_t = ckpt["round"] + 1
        load_vec(ckpt["lora_params"], ps)
        opt     = torch.optim.AdamW(ps, lr=LR_INNER)
        opt.load_state_dict(ckpt["opt_state"])
        history = ckpt["history"]
        print(f"Continuando do round {start_t}/{T}")
    else:
        start_t = 0
        opt     = torch.optim.AdamW(ps, lr=LR_INNER)
        history = []

    it = iter(loader)
    steps_per_round = R * H

    for t in tqdm(range(start_t, T), desc="Baseline"):
        acc_loss = 0.0
        for _ in range(steps_per_round):
            try:    ids, mask, labels = next(it)
            except: it = iter(loader); ids, mask, labels = next(it)
            opt.zero_grad()
            loss = step_loss(m, ids, mask, labels)
            loss.backward()
            opt.step()
            acc_loss += loss.item()

        acc = evaluate(m, val_loader)
        history.append({"train_loss": acc_loss / steps_per_round, "val_acc": acc})
        tqdm.write(f"  round={t:2d}  loss={history[-1]['train_loss']:.4f}  val_acc={acc:.4f}")

        torch.save({
            "round":       t,
            "lora_params": to_vec(ps),
            "opt_state":   opt.state_dict(),
            "history":     history,
        }, ckpt_path)

    return history

## 7. Executar Experimentos

In [ ]:
print('=== SparseLoCo + LoRA ===')
m_sloco = make_model()
hist_sloco = run_sparseloco(m_sloco, shards, val_loader)

print('\n=== Baseline: AdamW + LoRA ===')
m_base = make_model()
hist_base = run_baseline(m_base, central_loader, val_loader)

## 8. Resultados

In [ ]:
sloco_loss = [h['train_loss'] for h in hist_sloco]
sloco_acc  = [h['val_acc']   for h in hist_sloco]
base_loss  = [h['train_loss'] for h in hist_base]
base_acc   = [h['val_acc']   for h in hist_base]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(sloco_loss, 'o-', ms=4, label=f'SparseLoCo+LoRA (Top-{TOPK_K*100:.0f}%)')
ax1.plot(base_loss,  's-', ms=4, label='AdamW+LoRA (baseline)')
ax1.set_xlabel('Outer round'); ax1.set_ylabel('Training loss')
ax1.set_title('Training Loss'); ax1.legend()

ax2.plot(sloco_acc, 'o-', ms=4, label=f'SparseLoCo+LoRA (Top-{TOPK_K*100:.0f}%)')
ax2.plot(base_acc,  's-', ms=4, label='AdamW+LoRA (baseline)')
ax2.set_xlabel('Outer round'); ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy — MedQA'); ax2.legend()
ax2.set_ylim(0.1, 1.0)
ax2.axhline(y=1/N_LABELS, color='gray', linestyle='--', alpha=0.5, label=f'chance ({1/N_LABELS:.0%})')

plt.suptitle(f'SparseLoCo+LoRA vs AdamW+LoRA — {MODEL_ID.split("/")[-1]} | MedQA  (R={R}, H={H}, β={BETA})')
plt.tight_layout()
plt.savefig('resultados.png', dpi=150)
plt.show()

test_acc_sloco = evaluate(m_sloco, test_loader)
test_acc_base  = evaluate(m_base,  test_loader)

print(f"\n{'='*45}")
print(f"Avaliação final no TEST SET (holdout)")
print(f"{'='*45}")
print(f"SparseLoCo+LoRA:  {test_acc_sloco:.4f}")
print(f"AdamW+LoRA:       {test_acc_base:.4f}")
print(f"Chance level:     {1/N_LABELS:.4f}")